# FactSphere: Hallucination-Aware Agentic QA System

Janesh Danareddy | SRN: R23EA052 | Solo Project


In [ ]:
!pip install -q langchain langgraph langchain-anthropic \
    sentence-transformers chromadb wikipedia-api streamlit anthropic numpy scikit-learn pydantic


In [ ]:
import os
from google.colab import userdata

# Automatically import the ANTHROPIC_API_KEY from Colab's secret manager if available
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("API Key loaded successfully.")
except Exception:
    import getpass
    print("Please enter your Anthropic API Key manually:")
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass()


In [ ]:
import wikipediaapi
import chromadb
import numpy as np
import time
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field

# 1. Initialize models
print("Loading Models...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0
)
wiki_wiki = wikipediaapi.Wikipedia(user_agent='FactSphere/1.0 (janesh.danareddy@example.com)', language='en')

# 2. Knowledge Base Setup
print("Initializing Knowledge Base...")
chroma_client = chromadb.Client()

sample_docs = [
    # AI and machine learning concepts
    "Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. It enables computers to improve their performance on a specific task without being explicitly programmed.",
    "Neural networks are a series of algorithms that endeavor to recognize underlying relationships in a set of data. They are designed to mimic the way the human brain operates.",
    "Large Language Models (LLMs) are deep learning models trained on vast amounts of text data. They can generate human-like text and perform various natural language processing tasks.",
    # Famous scientists and inventors
    "Albert Einstein was a German-born theoretical physicist. He is best known for developing the theory of relativity, one of the two pillars of modern physics.",
    "Marie Curie was a Polish and naturalized-French physicist and chemist who conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize.",
    "Nikola Tesla was a Serbian-American inventor, electrical engineer, and futurist. He is best known for his contributions to the design of the modern alternating current (AC) electricity supply system.",
    # World history facts
    "The Industrial Revolution was the transition to new manufacturing processes in Great Britain, continental Europe, and the United States, that occurred during the period from around 1760 to about 1820-1840.",
    "The Great Wall of China is a series of fortifications that were built across the historical northern borders of ancient Chinese states. It was built to protect against various nomadic groups from the Eurasian Steppe.",
    # Basic geography
    "Mount Everest is Earth's highest mountain above sea level, located in the Mahalangur Himal sub-range of the Himalayas. The China-Nepal border runs across its summit point.",
    "The Amazon River in South America is the largest river by discharge volume of water in the world. It flows through Peru, Colombia, and Brazil before emptying into the Atlantic Ocean."
]

try:
    collection = chroma_client.create_collection(name="factsphere_docs")
    embeddings = embedder.encode(sample_docs)
    collection.add(
        embeddings=embeddings.tolist(),
        documents=sample_docs,
        ids=[f"doc_{i}" for i in range(len(sample_docs))]
    )
    print("Knowledge base created successfully with 10 chunks.")
except Exception as e:
    collection = chroma_client.get_collection(name="factsphere_docs")
    print("Knowledge base already exists.")


In [ ]:
class FactSphereState(TypedDict):
    query: str
    query_type: str
    reasoning: str
    retrieved_chunks: list
    similarity_scores: list
    generated_answer: str
    verifier_verdict: str
    verifier_reason: str
    avg_similarity: float
    iteration_count: int
    confidence_score: float
    decision: str
    final_response: str


In [ ]:
# --- Pydantic Models for Output Parsing ---
class PlannerOutput(BaseModel):
    query_type: str = Field(description="Must be factual, speculative, or ambiguous")
    reasoning: str

class VerifierOutput(BaseModel):
    verdict: str = Field(description="Must be SUPPORTED or NOT_SUPPORTED")
    reason: str

# --- 5 AGENTS ---

def planner_node(state: FactSphereState):
    system_prompt = "You are a query classifier. Classify the given query as factual, speculative, or ambiguous. Return JSON with keys: query_type, reasoning."
    structured_llm = llm.with_structured_output(PlannerOutput)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Query: {state['query']}")
    ]
    try:
        response = structured_llm.invoke(messages)
        state["query_type"] = response.query_type.lower()
        state["reasoning"] = response.reasoning
    except Exception as e:
        state["query_type"] = "ambiguous"
        state["reasoning"] = str(e)
    
    state["iteration_count"] = 0
    return state

def retriever_node(state: FactSphereState):
    query = state["query"]
    state["iteration_count"] += 1
    
    query_embedding = embedder.encode(query).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )
    
    chunks = results['documents'][0] if results['documents'] else []
    
    if chunks:
        chunk_embeddings = embedder.encode(chunks)
        sim_scores = cosine_similarity([query_embedding], chunk_embeddings)[0].tolist()
    else:
        sim_scores = []
        
    state["retrieved_chunks"] = chunks
    state["similarity_scores"] = sim_scores
    
    best_sim = max(sim_scores) if sim_scores else 0.0
    if best_sim < 0.4:
        page = wiki_wiki.page(query)
        if page.exists():
            wiki_summary = page.summary[0:500]
            state["retrieved_chunks"].append(f"Wikipedia Context: {wiki_summary}")
            state["similarity_scores"].append(best_sim)
            
    return state

def generator_node(state: FactSphereState):
    system_prompt = "You are a factual answer generator. Answer the question using ONLY the provided context. If the context does not support the answer, say UNSUPPORTED. Do not use any outside knowledge."
    context_text = "\n\n".join(state["retrieved_chunks"])
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Context:\n{context_text}\n\nQuestion: {state['query']}")
    ]
    response = llm.invoke(messages)
    state["generated_answer"] = response.content
    return state

def verifier_node(state: FactSphereState):
    system_prompt = "You are a fact verifier. Given an answer and context chunks, determine if the answer is fully supported by the context. Return JSON with keys: verdict (SUPPORTED or NOT_SUPPORTED), reason."
    structured_llm = llm.with_structured_output(VerifierOutput)
    context_text = "\n\n".join(state["retrieved_chunks"])
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Context:\n{context_text}\n\nAnswer: {state['generated_answer']}")
    ]
    try:
        response = structured_llm.invoke(messages)
        verdict = response.verdict
        state["verifier_reason"] = response.reason
    except Exception:
        verdict = "NOT_SUPPORTED"
        state["verifier_reason"] = "JSON Output Parsing Failed."
        
    ans_emb = embedder.encode([state["generated_answer"]])
    chunk_embs = embedder.encode(state["retrieved_chunks"]) if state["retrieved_chunks"] else []
    
    if len(chunk_embs) > 0:
        sims = cosine_similarity(ans_emb, chunk_embs)[0]
        avg_sim = float(np.mean(sims))
    else:
        avg_sim = 0.0
        
    state["avg_similarity"] = avg_sim
    
    if avg_sim < 0.35 or verdict == "NOT_SUPPORTED":
        state["verifier_verdict"] = "NOT_SUPPORTED"
    else:
        state["verifier_verdict"] = "SUPPORTED"
        
    return state

def confidence_node(state: FactSphereState):
    score = 1.0
    q_type = state.get("query_type", "factual")
    verdict = state.get("verifier_verdict", "SUPPORTED")
    iterations = state.get("iteration_count", 0)
    
    if q_type == "speculative":
        score -= 0.3
    elif q_type == "ambiguous":
        score -= 0.2
        
    if verdict == "NOT_SUPPORTED":
        score -= 0.3
        
    if iterations > 1:
        score -= 0.1 * (iterations - 1)
        
    score = max(0.0, min(1.0, score))
    state["confidence_score"] = score
    
    if score >= 0.6:
        state["decision"] = "ANSWER"
        state["final_response"] = state.get("generated_answer", "No answer generated.")
    elif score >= 0.3:
        state["decision"] = "CLARIFY"
        state["final_response"] = "Please provide more context or clarify your query."
    else:
        state["decision"] = "REFUSE"
        state["final_response"] = "No reliable evidence found for this query."
        
    return state


In [ ]:
def route_planner(state: FactSphereState):
    if state["query_type"] in ["speculative", "ambiguous"]:
        return "confidence"
    return "retriever"

def route_verifier(state: FactSphereState):
    if state["verifier_verdict"] == "NOT_SUPPORTED" and state["iteration_count"] < 3:
        return "retriever"
    return "confidence"

graph_builder = StateGraph(FactSphereState)
graph_builder.add_node("planner", planner_node)
graph_builder.add_node("retriever", retriever_node)
graph_builder.add_node("generator", generator_node)
graph_builder.add_node("verifier", verifier_node)
graph_builder.add_node("confidence", confidence_node)

graph_builder.set_entry_point("planner")
graph_builder.add_conditional_edges("planner", route_planner)
graph_builder.add_edge("retriever", "generator")
graph_builder.add_edge("generator", "verifier")
graph_builder.add_conditional_edges("verifier", route_verifier)
graph_builder.add_edge("confidence", END)

graph = graph_builder.compile()
print("Graph Compiled Successfully.")


In [ ]:
import time

test_queries = [
    # 5 Factual
    {"q": "What is machine learning?", "cat": "factual"},
    {"q": "Who was Marie Curie?", "cat": "factual"},
    {"q": "How long is the Great Wall of China?", "cat": "factual"}, # Should trigger wiki fallback
    {"q": "What is the largest river in South America?", "cat": "factual"},
    {"q": "Where is Mount Everest located?", "cat": "factual"},
    # 3 Ambiguous/Speculative
    {"q": "What is the best type of AI?", "cat": "ambiguous"},
    {"q": "Will AI take over the world by 2050?", "cat": "speculative"},
    {"q": "How does technology make us feel?", "cat": "ambiguous"},
    # 2 Fake
    {"q": "What year was the Moon made of cheese?", "cat": "fake"},
    {"q": "Who invented the gravity machine in 1980?", "cat": "fake"}
]

print(f"{'Query':<45} | {'Type':<11} | {'Verdict':<13} | {'Conf':<4} | {'Decision':<8} | Final Response")
print("-" * 120)

hallucination_count = 0
factual_total = 0
refusal_correct = 0
fake_total = 0
conf_scores = []
latencies = []

for item in test_queries:
    query = item["q"]
    true_cat = item["cat"]
    
    start_time = time.time()
    try:
        final_state = graph.invoke({"query": query})
    except Exception as e:
        final_state = {"decision": "ERROR", "final_response": str(e)}
        
    latency = time.time() - start_time
    latencies.append(latency)
    
    q_type = final_state.get('query_type', 'N/A')
    verdict = final_state.get('verifier_verdict', 'N/A')
    conf = final_state.get('confidence_score', 0.0)
    decision = final_state.get('decision', 'N/A')
    resp = final_state.get('final_response', '')[:50].replace('\n', ' ')
    
    conf_scores.append(conf)
    
    # Metrics Logic
    if true_cat == "factual":
        factual_total += 1
        if decision == "REFUSE" or "UNSUPPORTED" in final_state.get('generated_answer', ''):
            hallucination_count += 1
            
    if true_cat == "fake":
        fake_total += 1
        if decision == "REFUSE":
            refusal_correct += 1
            
    # Format and print
    q_trunc = (query[:42] + '...') if len(query) > 42 else query
    print(f"{q_trunc:<45} | {q_type:<11} | {verdict:<13} | {conf:.2f} | {decision:<8} | {resp}...")

print("\n" + "="*50)
print("EVALUATION METRICS")
print("="*50)
print(f"Hallucination Rate (Factual failures): {(hallucination_count/factual_total)*100 if factual_total > 0 else 0:.1f}%")
print(f"Refusal Accuracy (Correctly refused fake facts): {(refusal_correct/fake_total)*100 if fake_total > 0 else 0:.1f}%")
print(f"Average Confidence Score: {np.mean(conf_scores):.2f}")
print(f"Average Latency per Query: {np.mean(latencies):.2f} seconds")
